# Q1, analysis 1: perceptual RDM vs. the Kiani-Mur (KM) tree

Reads the tidy CSVs written by `python -m analysis.build_rdms.run_h1` (`analysis/results/hierarchy_comparison/`).

* **1a** RDM-level agreement between `behav_{pre,post}_simple` and `sem_km`: Kendall tau-a (primary, KM is heavily tied) and Spearman, with a NaN-aware image-relabeling permutation test, a participant bootstrap of the combined RDM, and per-participant model fit next to the lower noise-ceiling bound.
* **1b** Whole-tree agreement (Baker's gamma) between the exact KM dendrogram and agglomerative trees on MDS embeddings, plus per-KM-level HDBSCAN agreement.

Caveats carried through: Baker's gamma is dominated by coarse structure (most pairs merge near the root); HDBSCAN scores are over the images it clustered, read them with `frac_noise`; the permutation p floor is 1/(n_perm+1).

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from analysis.build_rdms.common import REPO_ROOT
from analysis.utils.plotting import role_palette

D = REPO_ROOT / "analysis" / "results" / "hierarchy_comparison"
T = {n: pd.read_csv(D / f"h1_{n}.csv") for n in
     ["summary", "permutation", "loo_fit", "bootstrap", "agglomerative", "gamma_shuffle", "hdbscan", "fdr"]}
COHORTS = ["pre", "post"]
LABEL = {"pre": "Pre-SHINE", "post": "Post-SHINE"}
COLOR = {c: v["violin_line"] if "violin_line" in v else list(v.values())[0] for c, v in role_palette(COHORTS).items()}
pd.set_option("display.width", 200)

## 1a. Confirmatory tests and summary

In [2]:
T['summary'].set_index('cohort').T.round(4)

cohort,pre,post
n_participants,79.0000,76.0000
n_boot,500.0000,500.0000
n_perm,1000.0000,1000.0000
combined_tau_a,0.2094,0.2100
combined_tau_a_ci_lo,0.1874,0.1868
combined_tau_a_ci_hi,0.2126,0.2162
combined_spearman,0.3176,0.3196
perm_p_tau_a,0.0010,0.0010
mean_model_tau_a,0.1856,0.1888
model_ci_lo,0.1711,0.1713


In [3]:
T['fdr'].round(4)

,cohort,test,p_value,p_fdr
0,pre,tau_a_permutation,0.001,0.001
1,pre,baker_gamma_average,0.001,0.001
2,pre,baker_gamma_ward,0.001,0.001
3,pre,baker_gamma_complete,0.001,0.001
4,post,tau_a_permutation,0.001,0.001
5,post,baker_gamma_average,0.001,0.001
6,post,baker_gamma_ward,0.001,0.001
7,post,baker_gamma_complete,0.001,0.001


Model fit is scored per participant against `sem_km`, next to the same participant's lower bound (their own RDM vs. the combined others). The model is noise-free, so it can exceed this noisy lower bound; the lower bound is a floor on the ceiling, not a cap on the model.

In [4]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[LABEL[c] for c in COHORTS], shared_yaxes=True)
for k, c in enumerate(COHORTS, start=1):
    d = T["loo_fit"].query("cohort == @c").dropna(subset=["model_kendall_a", "lower_kendall_a"])
    for name, col, x in [("KM model", "model_kendall_a", 0), ("Lower bound (others)", "lower_kendall_a", 1)]:
        fig.add_trace(go.Violin(y=d[col], x=[name] * len(d), name=name, box_visible=True, points="all",
                                line_color=COLOR[c], opacity=0.8, showlegend=False), row=1, col=k)
fig.update_yaxes(title_text="per-participant Kendall tau-a", row=1, col=1)
fig.update_layout(template="plotly_white", height=460, title="Per-participant model fit vs. lower noise bound")
fig

In [5]:
fig = go.Figure()
for c in COHORTS:
    b = T["bootstrap"].query("cohort == @c")["kendall_a"]
    fig.add_trace(go.Histogram(x=b, name=LABEL[c], marker_color=COLOR[c], opacity=0.7, nbinsx=30))
fig.update_layout(barmode="overlay", template="plotly_white", height=420,
                  xaxis_title="Kendall tau-a of combined RDM vs. KM (participant bootstrap)",
                  title="Sampling variability of the headline correlation")
fig

## 1b. Whole-tree agreement (agglomerative on MDS embeddings)

In [6]:
T['gamma_shuffle'].round(4)

,cohort,ndim,linkage,baker_gamma,null_mean,null_sd,p_value
0,pre,10,average,0.8065,0.0000,0.0040,0.001
1,pre,10,ward,0.8083,0.0000,0.0041,0.001
2,pre,10,complete,0.6668,0.0000,0.0039,0.001
3,post,10,average,0.6866,0.0001,0.0038,0.001
4,post,10,ward,0.6855,0.0001,0.0040,0.001
5,post,10,complete,0.6440,0.0001,0.0038,0.001


In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[LABEL[c] for c in COHORTS])
for k, c in enumerate(COHORTS, start=1):
    a = T["agglomerative"].query("cohort == @c").pivot(index="ndim", columns="linkage", values="baker_gamma")
    fig.add_trace(go.Heatmap(z=a.values, x=a.columns, y=[str(i) for i in a.index], zmin=0, zmax=1,
                             colorscale="Viridis", text=a.round(3).values, texttemplate="%{text}",
                             showscale=k == 2), row=1, col=k)
fig.update_yaxes(title_text="MDS dimensions", row=1, col=1)
fig.update_layout(template="plotly_white", height=420, title="Baker's gamma vs. KM tree, by MDS dimension and linkage")
fig

In [8]:
(T["agglomerative"].pivot_table(index=["cohort", "ndim"], columns="linkage", values="cophenetic_fidelity").round(3))

linkage      average  complete   ward
cohort ndim                          
post   2       0.717     0.688  0.717
       3       0.661     0.596  0.662
       5       0.680     0.481  0.677
       10      0.723     0.705  0.718
pre    2       0.718     0.715  0.718
       3       0.636     0.616  0.594
       5       0.631     0.466  0.622
       10      0.678     0.612  0.674

## 1b. HDBSCAN agreement with KM levels (ndim = 10)

In [9]:
h = T["hdbscan"].query("ndim == 10")
fig = make_subplots(rows=1, cols=3, subplot_titles=["Level 1: domain", "Level 2: mid-level", "Level 3: basic"], shared_yaxes=True)
for k, level in enumerate([1, 2, 3], start=1):
    for c in COHORTS:
        d = h.query("cohort == @c and km_level == @level")
        fig.add_trace(go.Scatter(x=d["min_cluster_size"], y=d["ari"], mode="lines+markers", name=LABEL[c],
                                 line_color=COLOR[c], showlegend=k == 1), row=1, col=k)
fig.update_xaxes(type="log", title_text="min_cluster_size")
fig.update_yaxes(title_text="ARI over clustered images", row=1, col=1)
fig.update_layout(template="plotly_white", height=420, title="HDBSCAN vs. KM categories")
fig

In [10]:
h.query("km_level == 1")[["cohort", "min_cluster_size", "n_clusters", "frac_noise"]].round(3)

,cohort,min_cluster_size,n_clusters,frac_noise
45,pre,2,67,0.188
48,pre,3,7,0.095
51,pre,5,6,0.095
54,pre,10,5,0.179
57,pre,20,5,0.248
105,post,2,88,0.163
108,post,3,21,0.254
111,post,5,7,0.263
114,post,10,5,0.261
117,post,20,4,0.247
